# Comparison of short-form versus sequential long-form transcription

## Imports

In [91]:
from pathlib import Path
import pandas as pd
from scipy.stats import spearmanr, binomtest

## Load Results

In [92]:
SHORT_RUN = Path("../results/asr/runs/decode_compare_small_short")

LONG_RUN = Path("../results/asr/runs/decode_compare_small_long")

short = pd.read_csv(SHORT_RUN / "scored.csv")

long = pd.read_csv(LONG_RUN / "scored.csv")

## Verify Experimental Conditions

In [93]:
short_ids = set(short["sample_id"])
long_ids = set(long["sample_id"])

assert short_ids == long_ids

assert short["sample_id"].is_unique
assert long["sample_id"].is_unique

In [94]:
print(len(short))
print(len(long))

370
370


## Keep Successful Samples Only
The samples being compared must have been successful in both

In [95]:
short_ok = short.loc[short["scoring_status"] == "ok"].copy()
long_ok = long.loc[long["scoring_status"] == "ok"].copy()

comparison = short_ok.merge(
    long_ok,
    on="sample_id",
    how="inner",
    suffixes=("_short", "_long"),
    validate="one_to_one",
)

comparison

,sample_id,dataset_short,model_name_short,decode_mode_short,transcription_status_short,scoring_status_short,scoring_error_type_short,scoring_error_message_short,reference_norm_short,hypothesis_norm_short,...,hypothesis_norm_long,n_punctuation_reference_long,n_punctuation_hypothesis_long,wer_long,substitutions_long,deletions_long,insertions_long,hits_long,n_reference_words_long,scored_at_utc_long
0,saa_english1,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.000000,0,0,0,69,69,2026-09-25T16:57:57.538791+00:00
1,saa_english10,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.028986,1,0,1,68,69,2026-09-25T16:57:57.540585+00:00
2,saa_english100,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.000000,0,0,0,69,69,2026-09-25T16:57:57.542461+00:00
3,saa_english102,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.000000,0,0,0,69,69,2026-09-25T16:57:57.543909+00:00
4,saa_english103,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.000000,0,0,0,69,69,2026-09-25T16:57:57.545555+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,saa_hindi3,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.000000,0,0,0,69,69,2026-09-25T16:57:58.203879+00:00
366,saa_hindi5,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.028986,1,0,1,68,69,2026-09-25T16:57:58.205968+00:00
367,saa_hindi6,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.000000,0,0,0,69,69,2026-09-25T16:57:58.207484+00:00
368,saa_hindi8,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,please call stella ask her to bring these thin...,0,0,0.014493,1,0,0,68,69,2026-09-25T16:57:58.209112+00:00


## Calculations
### Interpretation
* wer_difference > 0 : short-form was better
* wer_difference < 0 : long-form was better
* wer_difference = 0 : same WER

In [96]:
comparison["wer_difference"] = (comparison["wer_long"] - comparison["wer_short"])

comparison["same_wer"] = (comparison["wer_short"] == comparison["wer_long"])

comparison["same_transcript"] = (comparison["hypothesis_norm_short"]== comparison["hypothesis_norm_long"])

comparison["wer_changed"] = ~comparison["same_wer"]
comparison["transcript_changed"] = ~comparison["same_transcript"]

comparison

,sample_id,dataset_short,model_name_short,decode_mode_short,transcription_status_short,scoring_status_short,scoring_error_type_short,scoring_error_message_short,reference_norm_short,hypothesis_norm_short,...,deletions_long,insertions_long,hits_long,n_reference_words_long,scored_at_utc_long,wer_difference,same_wer,same_transcript,wer_changed,transcript_changed
0,saa_english1,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,69,69,2026-09-25T16:57:57.538791+00:00,0.000000,True,True,False,False
1,saa_english10,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,1,68,69,2026-09-25T16:57:57.540585+00:00,0.000000,True,True,False,False
2,saa_english100,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,69,69,2026-09-25T16:57:57.542461+00:00,0.000000,True,True,False,False
3,saa_english102,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,69,69,2026-09-25T16:57:57.543909+00:00,0.000000,True,True,False,False
4,saa_english103,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,69,69,2026-09-25T16:57:57.545555+00:00,0.000000,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,saa_hindi3,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,69,69,2026-09-25T16:57:58.203879+00:00,0.000000,True,True,False,False
366,saa_hindi5,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,1,68,69,2026-09-25T16:57:58.205968+00:00,0.000000,True,True,False,False
367,saa_hindi6,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,69,69,2026-09-25T16:57:58.207484+00:00,0.000000,True,True,False,False
368,saa_hindi8,saa,openai/whisper-small,short_form,ok,ok,NaN,NaN,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...,...,0,0,68,69,2026-09-25T16:57:58.209112+00:00,0.014493,False,False,True,True


## Summary

In [97]:
summary = {
    "n_samples": len(comparison),

    "mean_wer_short":
        comparison["wer_short"].mean(),

    "mean_wer_long":
        comparison["wer_long"].mean(),

    "median_wer_short":
        comparison["wer_short"].median(),

    "median_wer_long":
        comparison["wer_long"].median(),

    "mean_wer_difference":
        comparison["wer_difference"].mean(),

    "same_wer":
        comparison["same_wer"].sum(),

    "same_transcript":
        comparison["same_transcript"].sum(),

    "short_better":
        (
            comparison["wer_difference"] > 0
        ).sum(),

    "long_better":
        (
            comparison["wer_difference"] < 0
        ).sum(),
}

pd.Series(summary)

n_samples              370.000000
mean_wer_short           0.008578
mean_wer_long            0.009048
median_wer_short         0.000000
median_wer_long          0.000000
mean_wer_difference      0.000470
same_wer               338.000000
same_transcript        337.000000
short_better            20.000000
long_better             12.000000
dtype: float64

In [98]:
n_total = len(comparison)

n_different = (~comparison["same_transcript"]).sum()

pct_different = (100 * n_different / n_total)

print(
    f"Different Transcript: {n_different}/{n_total} "
    f"({pct_different:.2f}%)"
)

n_different_wer = (
    comparison["wer_short"]
    != comparison["wer_long"]
).sum()

pct_different_wer = (100 * n_different_wer / n_total)

print(
    f"Different WER: {n_different_wer}/{n_total} "
    f"({pct_different_wer:.2f}%)"
)

Different Transcript: 33/370 (8.92%)
Different WER: 32/370 (8.65%)


## Samples That Changed

In [99]:
changed = comparison.loc[~comparison["same_transcript"]].copy()

In [100]:
changed[
    [
        "sample_id",
        "wer_short",
        "wer_long",
        "wer_difference",
        "hypothesis_norm_short",
        "hypothesis_norm_long",
    ]
]

,sample_id,wer_short,wer_long,wer_difference,hypothesis_norm_short,hypothesis_norm_long
24,saa_english139,0.028986,0.057971,0.028986,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
31,saa_english151,0.043478,0.086957,0.043478,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
52,saa_english18,0.014493,0.000000,-0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
80,saa_english214,0.057971,0.072464,0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
83,saa_english219,0.000000,0.014493,0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
95,saa_english244,0.043478,0.028986,-0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
100,saa_english25,0.043478,0.057971,0.014493,please call stella esker to bring these things...,please call stella esker to bring these things...
113,saa_english272,0.000000,0.014493,0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
118,saa_english278,0.000000,0.014493,0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...
127,saa_english292,0.000000,0.014493,0.014493,please call stella ask her to bring these thin...,please call stella ask her to bring these thin...


## Accent Comparison
To check one accent isn't disproportionately larger

In [101]:
manifest = pd.read_csv("../data/manifests/study_under_30s.csv")

accent_comparison = comparison.merge(
    manifest[
        [
            "sample_id",
            "accent_group",
            "duration_s",
        ]
    ],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

accent_summary = (
    accent_comparison
    .groupby("accent_group")
    .agg(
        n_samples=("sample_id", "size"),
        n_changed=("transcript_changed", "sum"),
        wer_short=("wer_short", "mean"),
        wer_long=("wer_long", "mean"),
        mean_wer_difference=("wer_difference", "mean"),
    )
)

accent_summary["pct_changed"] = (
    100
    * accent_summary["n_changed"]
    / accent_summary["n_samples"]
)

accent_summary["wer_difference"] = (
    accent_summary["wer_long"]
    - accent_summary["wer_short"]
)

accent_summary["wer_difference_pp"] = (
    accent_summary["wer_difference"]
    * 100
)

accent_summary

,n_samples,n_changed,wer_short,wer_long,mean_wer_difference,pct_changed,wer_difference,wer_difference_pp
accent_group,,,,,,,,
american_english,331,19,0.006393,0.006568,0.000175,5.740181,0.000175,0.017514
amharic,9,3,0.056361,0.059581,0.003221,33.333333,0.003221,0.322061
hausa,5,1,0.008696,0.020290,0.011594,20.000000,0.011594,1.159420
hindi,15,5,0.011594,0.015459,0.003865,33.333333,0.003865,0.386473
kiswahili,6,3,0.033816,0.028986,-0.004831,50.000000,-0.004831,-0.483092
tigrigna,3,2,0.043478,0.043478,0.000000,66.666667,0.000000,0.000000
yoruba,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Pooled Accent Comparison

In [102]:
accent_comparison["accent_category"] = accent_comparison["accent_group"].where(
    accent_comparison["accent_group"] == "american_english",
    "non_american"
)

pooled_accent_summary = (
    accent_comparison
    .groupby("accent_category")
    .agg(
        n_samples=("sample_id", "size"),
        n_transcript_changed=("transcript_changed", "sum"),
        n_wer_changed=("wer_changed", "sum"),
        mean_wer_short=("wer_short", "mean"),
        mean_wer_long=("wer_long", "mean"),
        mean_wer_difference=("wer_difference", "mean"),
    )
)

pooled_accent_summary["pct_transcript_changed"] = (
    100
    * pooled_accent_summary["n_transcript_changed"]
    / pooled_accent_summary["n_samples"]
)

pooled_accent_summary["pct_wer_changed"] = (
    100
    * pooled_accent_summary["n_wer_changed"]
    / pooled_accent_summary["n_samples"]
)

pooled_accent_summary["wer_difference_pp"] = (
    100 * pooled_accent_summary["mean_wer_difference"]
)

pooled_accent_summary

,n_samples,n_transcript_changed,n_wer_changed,mean_wer_short,mean_wer_long,mean_wer_difference,pct_transcript_changed,pct_wer_changed,wer_difference_pp
accent_category,,,,,,,,,
american_english,331,19,19,0.006393,0.006568,0.000175,5.740181,5.740181,0.017514
non_american,39,14,13,0.027127,0.030100,0.002973,35.897436,33.333333,0.297287


### Interpretation

The short-duration comparison set is strongly dominated by American-English
recordings: 331 of the 370 samples (89.5%) belong to this group. Consequently,
the overall 91% agreement between decoding modes should not be interpreted as
evidence that the modes behave identically for every accent.

Transcript changes were more frequent in the pooled non-American subset.
However, only 39 non-American recordings were available for this comparison,
and the individual accent-group sample sizes were particularly small.
Accent-specific percentages should therefore be interpreted cautiously.

Despite the greater frequency of transcript changes in the non-American subset,
the mean WER increased by only approximately 0.30 percentage points under long-form decoding.”

In [103]:
reference_word_counts = (
    manifest["reference_transcript"]
    .str.split()
    .str.len()
)

reference_word_counts.value_counts()

reference_transcript
69    370
Name: count, dtype: int64

In [104]:
changed_wer = comparison.loc[comparison["wer_changed"]].copy()

N_REFERENCE_WORDS = 69

changed_wer["absolute_word_error_difference"] = (
    changed_wer["wer_difference"].abs()
    * N_REFERENCE_WORDS
).round().astype(int)

word_error_difference_summary = (
    changed_wer["absolute_word_error_difference"]
    .value_counts()
    .sort_index()
)

word_error_difference_summary

absolute_word_error_difference
1    25
2     5
3     1
4     1
Name: count, dtype: int64

In [105]:
n_changed_wer = len(changed_wer)

print(
    "WER differences of one word:",
    f"{(changed_wer['absolute_word_error_difference'] == 1).sum()}/{n_changed_wer}",
    f"({100 * (changed_wer['absolute_word_error_difference'] == 1).mean():.1f}%)"
)

print(
    "WER differences of at most two words:",
    f"{(changed_wer['absolute_word_error_difference'] <= 2).sum()}/{n_changed_wer}",
    f"({100 * (changed_wer['absolute_word_error_difference'] <= 2).mean():.1f}%)"
)

WER differences of one word: 25/32 (78.1%)
WER differences of at most two words: 30/32 (93.8%)


### Interpretation

Because every recording in this comparison uses the same 69-word reference
passage, the WER differences can also be interpreted in terms of the number of
word errors. Of the 32 recordings whose WER changed, 25 (78.1%) differed by
only one word and 30 (93.8%) differed by no more than two words.

Therefore, although sequential long-form decoding occasionally produces a
different transcription, the magnitude of most observed changes is small.

## Correlation between WER difference between transcription methods and the audio duration

In [106]:
short_transcriptions = pd.read_csv(SHORT_RUN / "transcriptions.csv")

comparison_durations = comparison.merge(
    short_transcriptions[
        [
            "sample_id",
            "processed_duration_s",
        ]
    ],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

comparison_durations[
    [
        "processed_duration_s",
        "wer_difference",
    ]
].corr(
    method="spearman"
)

,processed_duration_s,wer_difference
processed_duration_s,1.00000,0.04286
wer_difference,0.04286,1.00000


In [107]:
rho, p_value = spearmanr(
    comparison_durations["processed_duration_s"],
    comparison_durations["wer_difference"],
)

print(f"Spearman rho: {rho:.4f}")
print(f"p-value: {p_value:.4f}")

Spearman rho: 0.0429
p-value: 0.4111


In [108]:
n_short_better = (comparison["wer_difference"] > 0).sum()
n_long_better = (comparison["wer_difference"] < 0).sum()

sign_test = binomtest(
    k=n_short_better,
    n=n_short_better + n_long_better,
    p=0.5,
    alternative="two-sided",
)

print(f"Short-form better: {n_short_better}")
print(f"Long-form better: {n_long_better}")
print(f"Exact sign-test p-value: {sign_test.pvalue:.4f}")

Short-form better: 20
Long-form better: 12
Exact sign-test p-value: 0.2153


In [109]:
comparison_durations["duration_bin"] = pd.cut(
    comparison_durations["processed_duration_s"],
    bins=[0, 10, 20, 25, 30],
)

duration_bin_summary = (
    comparison_durations
    .groupby("duration_bin", observed=True)
    .agg(
        n_samples=("sample_id", "size"),
        mean_wer_short=("wer_short", "mean"),
        mean_wer_long=("wer_long", "mean"),
        mean_wer_difference=("wer_difference", "mean"),
    )
)

duration_bin_summary["wer_difference_pp"] = (
    100 * duration_bin_summary["mean_wer_difference"]
)

duration_bin_summary

,n_samples,mean_wer_short,mean_wer_long,mean_wer_difference,wer_difference_pp
duration_bin,,,,,
"(10, 20]",79,0.005320,0.005320,0.000000,0.000000
"(20, 25]",223,0.006954,0.007409,0.000455,0.045493
"(25, 30]",68,0.017690,0.018755,0.001066,0.106564


In [110]:
duration_with_accent = comparison_durations.merge(
    manifest[
        [
            "sample_id",
            "accent_group",
        ]
    ],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

duration_with_accent["duration_bin"] = pd.cut(
    duration_with_accent["processed_duration_s"],
    bins=[0, 10, 20, 25, 30],
)

duration_accent_summary = (
    duration_with_accent
    .groupby("duration_bin", observed=True)
    .agg(
        n_samples=("sample_id", "size"),
        n_american=(
            "accent_group",
            lambda x: (x == "american_english").sum(),
        ),
    )
)

duration_accent_summary["pct_american"] = (
    100
    * duration_accent_summary["n_american"]
    / duration_accent_summary["n_samples"]
)

duration_accent_summary

,n_samples,n_american,pct_american
duration_bin,,,
"(10, 20]",79,79,100.000000
"(20, 25]",223,209,93.721973
"(25, 30]",68,43,63.235294


### Interpretation

There was negligible monotonic association between recording duration and the
difference in WER between decoding modes (Spearman rho = 0.043, p = 0.411).
Therefore, there is no evidence within the recordings below 30 seconds that the
difference between sequential long-form and short-form decoding systematically
increases with recording duration.

The duration bins are not independent of accent composition. All samples in
the 10--20 second interval are American English, whereas the proportion of
American-English recordings decreases in the longer-duration bins. Therefore,
differences between duration bins should not be interpreted as effects of
duration alone.

Among the 32 recordings for which WER differed, short-form produced the lower
WER in 20 cases and sequential long-form produced the lower WER in 12 cases.
An exact sign test did not identify a significant directional preference
(p = 0.215). Thus, the experiment does not provide evidence that one decoding
mode systematically produces lower WER than the other. 

## Full Study Manifest Duration Distribution

The decode-mode comparison can only be performed on recordings shorter than or
equal to 30 seconds, since short-form decoding is not applicable to longer
recordings.    
The duration distribution of the complete study manifest
is examined by accent to determine whether decode mode would otherwise vary
systematically across accent groups.

In [111]:
full_manifest = pd.read_csv("../data/manifests/study_manifest.csv")

duration_by_accent = (
    full_manifest
    .groupby("accent_group")
    .agg(
        n_total=("sample_id", "size"),
        n_le_30=("duration_s", lambda x: (x <= 30).sum()),
        n_gt_30=("duration_s", lambda x: (x > 30).sum()),
    )
)

duration_by_accent["pct_le_30"] = (
    100
    * duration_by_accent["n_le_30"]
    / duration_by_accent["n_total"]
)

duration_by_accent["pct_gt_30"] = (
    100
    * duration_by_accent["n_gt_30"]
    / duration_by_accent["n_total"]
)

duration_by_accent

,n_total,n_le_30,n_gt_30,pct_le_30,pct_gt_30
accent_group,,,,,
american_english,340,331,9,97.352941,2.647059
amharic,20,9,11,45.000000,55.000000
hausa,9,5,4,55.555556,44.444444
hindi,18,15,3,83.333333,16.666667
kiswahili,9,6,3,66.666667,33.333333
tigrigna,8,3,5,37.500000,62.500000
yoruba,5,1,4,20.000000,80.000000


In [112]:
full_manifest["accent_category"] = full_manifest["accent_group"].where(
    full_manifest["accent_group"] == "american_english",
    "non_american"
)

duration_pooled = (
    full_manifest
    .groupby("accent_category")
    .agg(
        n_total=("sample_id", "size"),
        n_le_30=("duration_s", lambda x: (x <= 30).sum()),
        n_gt_30=("duration_s", lambda x: (x > 30).sum()),
    )
)

duration_pooled["pct_gt_30"] = (
    100
    * duration_pooled["n_gt_30"]
    / duration_pooled["n_total"]
)

duration_pooled

,n_total,n_le_30,n_gt_30,pct_gt_30
accent_category,,,,
american_english,340,331,9,2.647059
non_american,69,39,30,43.478261


### Interpretation

The complete SAA study manifest contains 409 recordings, of which 370 (90.5%) are
30 seconds or shorter. However, this overall percentage masks a substantial
difference in duration distribution between accent groups.

Only 2.6% of American-English recordings exceed 30 seconds, whereas 43.5% of
the non-American recordings exceed 30 seconds. Therefore, using short-form
decoding for recordings below 30 seconds and long-form decoding for recordings
above 30 seconds would cause decode mode to vary systematically with accent.
Using sequential long-form decoding for all recordings keeps the transcription method 
consistent across all accent groups.

## Floor-Effect Check

In [113]:
zero_wer_summary = pd.Series({
    "short_zero_wer_n":
        (comparison["wer_short"] == 0).sum(),

    "short_zero_wer_pct":
        100 * (comparison["wer_short"] == 0).mean(),

    "long_zero_wer_n":
        (comparison["wer_long"] == 0).sum(),

    "long_zero_wer_pct":
        100 * (comparison["wer_long"] == 0).mean(),
})

zero_wer_summary

short_zero_wer_n      267.000000
short_zero_wer_pct     72.162162
long_zero_wer_n       265.000000
long_zero_wer_pct      71.621622
dtype: float64

In [114]:
zero_wer_by_accent = (
    accent_comparison
    .assign(
        short_zero=lambda x: x["wer_short"] == 0,
        long_zero=lambda x: x["wer_long"] == 0,
    )
    .groupby("accent_category")
    .agg(
        n_samples=("sample_id", "size"),
        pct_zero_short=("short_zero", "mean"),
        pct_zero_long=("long_zero", "mean"),
    )
)

zero_wer_by_accent[
    ["pct_zero_short", "pct_zero_long"]
] *= 100

zero_wer_by_accent

,n_samples,pct_zero_short,pct_zero_long
accent_category,,,
american_english,331,75.830816,75.830816
non_american,39,41.025641,35.897436


### Interpretation

A substantial floor effect is present in the overall comparison, with 72.2%
of recordings obtaining a WER of zero under short-form decoding and 71.6%
under sequential long-form decoding.

However, this effect is largely driven by the American-English subset, for
which 75.8% of recordings obtained zero WER under both decoding modes.
The proportion of zero-WER recordings was considerably lower for the pooled
non-American subset (41.0% for short-form and 35.9% for long-form).

Since many recordings have a WER of zero, the overall comparison may hide some differences between the two decoding methods, especially for American English. The non-American recordings have more non-zero WER values, so they give a clearer comparison. Looking at the actual normalised transcripts also helps identify differences even when the final WER stays the same.